# Code-Constrained Plan Gen — Kaggle Smoke Test

**Purpose**: verify the full Stage 1+2 pipeline works in a fresh Kaggle environment, end-to-end, with the same numbers we get locally.

## Setup before Run All
1. **Settings** (right sidebar):
   - **Internet**: ON
   - **Accelerator**: NONE (CPU is enough for the smoke test; save GPU quota for training)
2. **+ Add data** (right sidebar): search for `modified-swiss-dwellings` and add the dataset by `caspervanengelenburg`.
3. Then `Run All`.

Expected runtime: 5–8 minutes.

## 1. Clone the repository and install dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/hemekci/code-constrained-plan-gen.git"
REPO_DIR = "/kaggle/working/code-constrained-plan-gen"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Repo:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

In [ ]:
# Install deps. shapely / networkx / torch / pandas already present on Kaggle;
# we only need to make sure pytest is available and our src/ is importable.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "shapely>=2.0", "networkx>=3.0"], check=True)

import importlib
for mod in ("shapely", "networkx", "torch", "pandas", "pytest"):
    m = importlib.import_module(mod)
    print(f"  {mod:10s} {getattr(m, '__version__', '?')}")

## 2. Wire up MSD data — symlink the Kaggle dataset into the expected location

In [ ]:
from pathlib import Path

MSD_KAGGLE = Path("/kaggle/input/modified-swiss-dwellings")
MSD_LOCAL = Path(REPO_DIR) / "data" / "MSD" / "raw"

if not MSD_KAGGLE.exists():
    raise SystemExit(
        "Add the dataset 'caspervanengelenburg/modified-swiss-dwellings' via\n"
        "the right sidebar (+ Add data) before Run All."
    )

MSD_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if MSD_LOCAL.exists() or MSD_LOCAL.is_symlink():
    if MSD_LOCAL.is_symlink():
        MSD_LOCAL.unlink()
    elif not any(MSD_LOCAL.iterdir()):
        MSD_LOCAL.rmdir()
if not MSD_LOCAL.exists():
    MSD_LOCAL.symlink_to(MSD_KAGGLE)
print("MSD pointed at:", MSD_LOCAL.resolve())
print("Top-level entries:", sorted(p.name for p in MSD_LOCAL.iterdir())[:8])

## 3. Run the unit-test suite

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise SystemExit("Tests failed.")

## 4. Smoke-scan: 50 MSD plans × 9 jurisdictions × 6 rules

This re-runs the same compliance scan we use locally but with `--limit 50` for speed (full 200-floor scan locally takes a few seconds, but Kaggle's CPU is slower).

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/baseline_compliance.py",
        "--dataset", "msd",
        "--split", "train",
        "--limit", "50",
        "--with-doors",
        "--out", "/kaggle/working/baseline_compliance_kaggle.json",
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)

## 5. Inspect the output JSON

In [ ]:
import json

with open("/kaggle/working/baseline_compliance_kaggle.json") as fh:
    summary = json.load(fh)

print(f"Dataset: {summary['dataset']} / {summary['split']}")
print(f"Floors scanned: {summary['n_floors_scanned']}")
print(f"With doors: {summary['with_doors']}")
print()
for jcode, jdata in summary["jurisdictions"].items():
    rates = jdata["rule_pass_rates"]
    print(f"=== {jcode} ===")
    for rule_name, r in rates.items():
        print(f"  {rule_name:30s} {r['pass']:3d}/{r['total']:3d}  ({100*r['rate']:5.1f}%)")
    print(f"  ALL PASS                       {jdata['n_all_passed']:3d}/{summary['n_floors_scanned']:3d}  ({100*jdata['all_pass_rate']:5.1f}%)")
    print()

## 6. Smoke-test the differentiable energy + guidance loop

In [ ]:
import sys
sys.path.insert(0, str(Path(REPO_DIR) / "src"))

import torch
from code_module.differentiable import door_width_energy
from code_module.guided_sampling import universal_guidance_sample
from model_module import MockDiffusionBackbone

target = torch.tensor([[[0.0, 0.0], [0.70, 0.0], [0.70, 0.10], [0.0, 0.10]]], dtype=torch.float64)
backbone = MockDiffusionBackbone(target=target, n_steps=30)
torch.manual_seed(0)
x_T = torch.randn_like(target)

def fn(x):
    return door_width_energy(x, min_width_m=0.90)

_, history_unguided = universal_guidance_sample(backbone, x_T.clone(), fn, guidance_scale=0.0)
_, history_guided = universal_guidance_sample(backbone, x_T.clone(), fn, guidance_scale=0.5)

print(f"Unguided final energy: {history_unguided.energies[-1]:.4f}")
print(f"Guided   final energy: {history_guided.energies[-1]:.4f}")
assert history_guided.energies[-1] < history_unguided.energies[-1], \
    "Guidance should reduce energy along the trajectory"
print("OK — guidance reduces energy.")

## 7. Done

If every cell ran without error, the full Stage 1+2 pipeline works in Kaggle:

- 26 unit tests passing
- 6-rule × 9-jurisdiction baseline scan reproduces the local numbers
- Differentiable energy + universal-guidance loop runs end-to-end

Next notebook: `kaggle_train_housediffusion.ipynb` — actual GPU training.